## Install dependencies

In [1]:
!pip install -q \
    "numpy>=1.22,<2.1" \
    "langchain==0.3.25" \
    "langchain-core==0.3.62" \
    "langchain-text-splitters==0.3.8" \
    "langchain-google-genai==2.1.5" \
    "langchain-chroma==0.2.4" \
    "langchain-huggingface==0.1.2" \
    "chromadb>=1.0.9" \
    "rank-bm25==0.2.2" \
    "sentence-transformers>=2.6.0" \
    "gradio>=4.40,<6.0" \
    "gitpython>=3.1"


## Configure Gemini API Key

In [2]:
## Setup api keys
import os

# Suppress ChromaDB telemetry warnings (harmless posthog SDK incompatibility)
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY_IMPL"] = "none"

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab secrets ✓")
except Exception:
    if "GOOGLE_API_KEY" not in os.environ:
        from getpass import getpass
        os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")
    print("API key set ✓")

API key set ✓


## Temporary store git repo

In [3]:
#Import codebases from git
import shutil
from pathlib import Path
import git

# Change this to point at your own repo any time.
REPO_URL = "https://github.com/Priyabrata111/Ecommerce-App"
LOCAL_PATH = Path("/tmp/repo")

# Wipe any previous clone
if LOCAL_PATH.exists():
    shutil.rmtree(LOCAL_PATH)

git.Repo.clone_from(REPO_URL, LOCAL_PATH, depth=1)
print(f"Cloned into {LOCAL_PATH}")
print(f"Files: {sum(1 for _ in LOCAL_PATH.rglob('*') if _.is_file())}")

Cloned into /tmp/repo
Files: 107


## View the Directory

In [4]:
def tree(p: Path, prefix: str = "", depth: int = 2) -> None:
    if depth < 0:
        return
    entries = sorted([e for e in p.iterdir() if not e.name.startswith(".")])
    for i, entry in enumerate(entries):
        is_last = i == len(entries) - 1
        connector = "└── " if is_last else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir():
            extension = "    " if is_last else "│   "
            tree(entry, prefix + extension, depth - 1)

tree(LOCAL_PATH)

├── client
│   └── build
│       ├── asset-manifest.json
│       ├── favicon.ico
│       ├── images
│       ├── index.html
│       ├── logo192.png
│       ├── logo512.png
│       ├── manifest.json
│       ├── robots.txt
│       └── static
├── config
│   └── db.js
├── controllers
│   ├── authController.js
│   ├── categoryController.js
│   └── productController.js
├── helpers
│   └── authHelper.js
├── middlewares
│   └── authMiddleware.js
├── models
│   ├── categoryModel.js
│   ├── orderModel.js
│   ├── productModel.js
│   └── userModel.js
├── package-lock.json
├── package.json
├── routes
│   ├── authRoute.js
│   ├── categoryRoutes.js
│   └── productRoutes.js
└── server.js


## Work with mutiple repos

In [5]:
REPOS = [
    {
        "name": "Ecommerce-App",
        "url": "https://github.com/Priyabrata111/Ecommerce-App"
    },
    {
        "name": "systemC",
        "url": "https://github.com/Priyabrata111/systemC"
    },
    {
        "name": "genAI_Learning",
        "url": "https://github.com/Priyabrata111/GenAI_Learning"
    },
    {
        "name": "Simon-Game",
        "url": "https://github.com/Priyabrata111/Simon-Game-Chellange"
    },
    {
        "name": "inotebook",
        "url": "https://github.com/Priyabrata111/inotebook"
    },
]


## Import all Repos in local system

In [6]:
#Import codebases from git
import shutil
from pathlib import Path
import git


## Store here
ROOT_DIR = Path("/tmp/repos")

for repo in REPOS:
    repo_path = ROOT_DIR / repo["name"]

    if repo_path.exists():
        shutil.rmtree(repo_path)

    git.Repo.clone_from(
        repo["url"],
        repo_path,
        depth=1
    )

## Check if repos are copied correctly

In [8]:
from pathlib import Path

for p in Path("/tmp/repos").iterdir():
    print(p)
    tree(p)

/tmp/repos/systemC
├── UST
│   └── Basics
│       ├── AT_tlm.cpp
│       ├── event.cpp
│       ├── handShake.cpp
│       ├── interrupts_method.cpp
│       ├── interrupts_thread.cpp
│       ├── multiTransaction.cpp
│       ├── producer_consumer.cpp
│       ├── raceAround.cpp
│       ├── tlm.cpp
│       └── tlm2.cpp
└── vConnect
    ├── EthernetController
    │   ├── EthernetFrame.h
    │   ├── receiver.h
    │   ├── target.h
    │   ├── testBench.cpp
    │   └── vECU.h
    ├── LearningAll.cpp
    ├── LearningBasics.cpp
    ├── TLM
    │   ├── LT.h
    │   ├── Packet.h
    │   ├── mainLT.cpp
    │   ├── myCPU.h
    │   ├── myCPUPacket.h
    │   ├── myMemory.h
    │   ├── myMemoryPacket.h
    │   ├── testBench02.cpp
    │   └── testbench.cpp
    ├── designCounterMethod.cpp
    └── designCounterThread.cpp
/tmp/repos/inotebook
├── README.md
├── backend
│   ├── db.js
│   ├── index.js
│   ├── middlewares
│   │   └── fetchuser.js
│   ├── models
│   │   ├── Note.js
│   │   └── User.js
│   ├── p

## Load the repos as langchain documents

In [9]:


from pathlib import Path
from typing import List
from langchain_core.documents import Document
# Extensions relevant for MERN / Node.js / React
CODE_EXTENSIONS = {
      # C/C++
    ".c",
    ".cc",
    ".cpp",
    ".cxx",
    ".h",
    ".hh",
    ".hpp",
    ".hxx",

    # SystemC / TLM
    ".tpp",

    # Python
    ".py",
    ".ipynb",

    # JavaScript / TypeScript
    ".js",
    ".jsx",
    ".ts",
    ".tsx",

    # Config / Docs
    ".json",
    ".yaml",
    ".yml",
    ".xml",
    ".toml",
    ".ini",
    ".md",
    ".txt",

    # Web
    ".css",
    ".html",
}

# Folders to ignore
SKIP_DIRS = {
    ".git",
    "__pycache__",
    "node_modules",
    "dist",
    ".next",
    ".cache",
}
# Function to load the git repo as langchain model

def load_repo_documents(repo_path: Path,repo_name: str) -> List[Document]:
    docs = []

    for path in repo_path.rglob("*"):

        # Skip directories
        if not path.is_file():
            continue

        # Skip unwanted folders
        if any(part in SKIP_DIRS for part in path.parts):
            continue

        # Skip unsupported file types
        if path.suffix.lower() not in CODE_EXTENSIONS:
            continue

        try:
            text = path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            continue

        rel = path.relative_to(repo_path)

        docs.append(
            Document(
                page_content=text,
                metadata={
                "repo": repo_name,
                "source": str(rel),
                "extension": path.suffix,
},
            )
        )

    return docs


# Function call to load the repo as langchain documents
all_docs = []
for repo in REPOS:
    repo_path = ROOT_DIR / repo["name"]

    docs = load_repo_documents(
        repo_path,
        repo_name=repo["name"]
    )

    all_docs.extend(docs)

    print(f"Loaded {len(docs)} files from {repo['name']}")


print(f"\nTotal files loaded: {len(all_docs)}")

for d in all_docs[:10]:
    print(
        f"[{d.metadata['repo']}] "
        f"{d.metadata['source']:40s} "
        f"({len(d.page_content)} chars)"
    )

Loaded 24 files from Ecommerce-App
Loaded 28 files from systemC
Loaded 12 files from genAI_Learning
Loaded 3 files from Simon-Game
Loaded 28 files from inotebook

Total files loaded: 95
[Ecommerce-App] package.json                             (854 chars)
[Ecommerce-App] server.js                                (1440 chars)
[Ecommerce-App] package-lock.json                        (135391 chars)
[Ecommerce-App] models/productModel.js                   (697 chars)
[Ecommerce-App] models/userModel.js                      (629 chars)
[Ecommerce-App] models/orderModel.js                     (496 chars)
[Ecommerce-App] models/categoryModel.js                  (273 chars)
[Ecommerce-App] helpers/authHelper.js                    (384 chars)
[Ecommerce-App] middlewares/authMiddleware.js            (829 chars)
[Ecommerce-App] controllers/authController.js            (6383 chars)
